Session 2 RQ2 preparation v2: unchanged data, 58 starter tests and four baselines. Independent of model/network preflight.

In [ ]:
from pathlib import Path
import sys, json, hashlib, subprocess, time, zipfile, os
from datetime import datetime, timezone
ROOT = Path.cwd()
OWN = ROOT / "llm4rec_s2_rq2"
INPUT = OWN / "inputs"
expected = {
 "FAR-Trans.zip": "d0d6cfe027732bab45fdcf9af80eff4fefddc789aedae9bdce0d3d6cfd05cfb6",
 "starter.py": "c19b21adeb824737d09835fb5b8818e2d310fee83e5af909eaf8897ea1705b24",
 "extensions.py": "c80ddbe27846d2f0ca49ce4a66365f4951db2a226b141f5ce1b3b3fdc94114a9",
 "test_starter.py": "a3a86f98d67f2416a99b1ea70e495ab3b82b3a1e5a8f4a908f0a6bae8e53d1c1",
 "llm_rerank_original.py": "d7a34c1f9f715d7437ca5b2cc7989dbff12348602495b8a427cf8e56fa7007ba"}
actual = {n: hashlib.sha256((INPUT / n).read_bytes()).hexdigest() for n in expected}
assert actual == expected
(OWN / "input-hashes.json").write_text(json.dumps(actual, indent=2) + "\n")
with zipfile.ZipFile(INPUT / "FAR-Trans.zip") as z:
    members = z.namelist()
    assert all(not Path(n).is_absolute() and ".." not in Path(n).parts for n in members)
    z.extractall(OWN / "data")
data = [p.parent for p in (OWN / "data").rglob("transactions.csv")]
assert len(data) == 1
DATA = data[0]
(OWN / "data-path.json").write_text(json.dumps({"relative": str(DATA.relative_to(ROOT)), "archive_members": members}, indent=2))
test_cmd = [sys.executable, "-m", "unittest", "-v", "test_starter"]
t0 = time.perf_counter()
r = subprocess.run(test_cmd, cwd=INPUT, text=True, capture_output=True)
(OWN / "starter-tests.log").write_text(r.stdout + r.stderr)
assert r.returncode == 0, r.stdout + r.stderr
assert "Ran 58 tests" in r.stdout + r.stderr
manifest = {"tests": {"command": test_cmd, "cwd": str(INPUT), "returncode": r.returncode, "seconds": time.perf_counter()-t0, "count": 58}}
cmd = [sys.executable, str(INPUT / "starter.py"), "run", "--data-dir", str(DATA), "--output-dir", str(OWN / "baseline")]
t0 = time.perf_counter()
r = subprocess.run(cmd, text=True, capture_output=True)
(OWN / "baseline-console.log").write_text(r.stdout + r.stderr)
assert r.returncode == 0, r.stdout + r.stderr
manifest["baseline"] = {"command": cmd, "seconds_whole_starter_run_including_all_four_methods_and_io": time.perf_counter()-t0, "returncode": r.returncode}
protocol = json.loads((OWN / "baseline/protocol.json").read_text())
assert protocol["protocol_hash"].startswith("50efd4aeb415")
assert len(protocol["users"]) == 100
manifest["protocol_hash"] = protocol["protocol_hash"]
manifest["at_utc"] = datetime.now(timezone.utc).isoformat()
(OWN / "preparation-manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print(r.stdout)
print(json.dumps(manifest, indent=2))
